# Wariant B: Prompt-based learning
Wybierz jedno proste zadanie (np. klasyfikacja sentymentu lub rozpoznawanie encji) i zaprojektuj kilka różnych promptów opisujących to samo zadanie w inny sposób.

Przetestuj model na tym samym zbiorze przykładów, zmieniając wyłącznie treść promptu.

Porównaj wyniki jakościowo, ze szczególnym uwzględnieniem stabilności odpowiedzi i podatności modelu na zmianę sformułowania polecenia.

Rozwiązanie prześlij jako plik <twoje_nazwisko>_B.ipynb zawierający kompletny kod i wnioski.

In [4]:
from transformers import pipeline

clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Device set to use mps:0


In [5]:
hard_texts = [
    "Oh great, another 'bold' remake that plays it safe in every possible way.",
    "If you enjoy watching paint dry, you'll love this masterpiece.",

    "It has moments of genuine charm, though the story keeps tripping over clichés.",
    "Not perfect by any means, yet I found myself smiling more often than I expected.",

    "It isn't bad, but it never becomes truly good either.",
    "I can't say I hated it, which is about the nicest thing I can say.",

    "It's watchable, I suppose, but largely forgettable.",
    "Competent and polished, but emotionally hollow.",

    "Better than the trailer suggests, though still far from memorable.",
    "It starts brilliantly, then slowly collapses under its own weight.",

    "The film is so relentlessly quirky that it becomes exhausting rather than endearing.",
    "A quiet little drama that sneaks up on you and lands with surprising warmth.",
]

texts = hard_texts

labels = ["positive", "negative"]

prompts = [
    "This text expresses {} sentiment.",
    "Overall, the review is {}.",
    "The author feels {} about the movie.",
    "Sentiment: {}.",
]

In [10]:
from transformers import pipeline

gen = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=5
)

Device set to use mps:0


In [11]:
prompts = [
    lambda t: f"Classify the sentiment of the text as positive or negative.\nText: {t}\nAnswer:",
    lambda t: f"Is the following movie review positive or negative?\nReview: {t}\nLabel:",
    lambda t: f"Sentiment analysis task.\nInput: {t}\nOutput (positive/negative):",
    lambda t: f"Here is an example of negative review: 'The premise is intriguing, but the execution is uneven and the finale feels rushed'.\n Here is an example of positive review: 'It's not just good — it's surprisingly thoughtful and well-acted.'\n Now answer is the review is positive or negative. {t}"
]

In [12]:
for t in texts:
    print("TEXT:", t)
    for p in prompts:
        inp = p(t)
        out = gen(inp)[0]["generated_text"]
        print("\nPROMPTED INPUT:\n", inp)
        print("MODEL OUTPUT:\n", out)

TEXT: Oh great, another 'bold' remake that plays it safe in every possible way.

PROMPTED INPUT:
 Classify the sentiment of the text as positive or negative.
Text: Oh great, another 'bold' remake that plays it safe in every possible way.
Answer:
MODEL OUTPUT:
 positive

PROMPTED INPUT:
 Is the following movie review positive or negative?
Review: Oh great, another 'bold' remake that plays it safe in every possible way.
Label:
MODEL OUTPUT:
 positive

PROMPTED INPUT:
 Sentiment analysis task.
Input: Oh great, another 'bold' remake that plays it safe in every possible way.
Output (positive/negative):
MODEL OUTPUT:
 It's a

PROMPTED INPUT:
 Here is an example of negative review: 'The premise is intriguing, but the execution is uneven and the finale feels rushed'.
 Here is an example of positive review: 'It's not just good — it's surprisingly thoughtful and well-acted.'
 Now answer is the review is positive or negative. Oh great, another 'bold' remake that plays it safe in every possible wa

Prompty 1 i 2 są do siebie bardzo podobne i zwykle dają identyczną klasę
Dla większości tekstów P1 i P2 zwracają to samo ==> małe parafrazy instrukcji (classify vs is… positive or negative) nie wpływają jakoś mocno

P3 jest niestabilny i często w ogóle nie daje etykiety
W wielu przypadkach odpowiedź jest niepoprawna formatowo: "It's a", "sexy", "abrasive", "if it's", "I'm a".
To znaczy, że ten prompt zamiast wybrać label, zaczyna coś generować: opis lub losowe słowa

Prompt 4 z przykładami potrafi troche liepiej wykrywac sentyment i zmieniać klasę w porównaniu do P1/P2
“Oh great, another 'bold' remake…”: P1/P2 = positive, P4 = Negative
“Better than the trailer suggests…”: P1/P2 = positive, P4 = Negative